In [1]:
!pip install interpret --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 31.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.1 MB/s eta 0:00:00


In [2]:
!pip install wget --quiet

  Preparing metadata (setup.py) ... done


In [3]:
!pip install pgeocode --quiet

In [4]:
!pip install fastFM --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [18]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 49.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
salib 1.5.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have num

Imports

In [1]:
# Manipulacao de dados
import pandas as pd
import numpy as np

# Preprocessamento e avaliacao
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import NearestNeighbors
from scipy import sparse as sps
import pgeocode

# Modelo-base de analise: Factorization Machines
from fastFM.als import FMRegression

# Metodo de analise global: EBM
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show

# Utils
from tqdm import tqdm


# RNG
rng = np.random.RandomState(42)

### Carregamento dos dados e preprocessamento

In [2]:
# Download dos dados do movielens-100k
!wget http://files.grouplens.org/datasets/movielens/ml-100k/u.data -O u.data
!wget http://files.grouplens.org/datasets/movielens/ml-100k/u.item -O u.item
!wget http://files.grouplens.org/datasets/movielens/ml-100k/u.user -O u.user

--2025-11-30 22:28:40--  http://files.grouplens.org/datasets/movielens/ml-100k/u.data
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-100k/u.data [following]
--2025-11-30 22:28:40--  https://files.grouplens.org/datasets/movielens/ml-100k/u.data
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1979173 (1.9M)
Saving to: ‘u.data’

u.data              100%[===================>]   1.89M  7.54MB/s    in 0.3s    

2025-11-30 22:28:40 (7.54 MB/s) - ‘u.data’ saved [1979173/1979173]

--2025-11-30 22:28:40--  http://files.grouplens.org/datasets/movielens/ml-100k/u.item
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (f

In [3]:
other_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL']
genre_cols = ['Unknown','Action','Adventure','Animation','Children','Comedy',
              'Crime', 'Documentary','Drama','Fantasy','Film-Noir','Horror',
              'Musical', 'Mystery','Romance','Sci-Fi','Thriller','War','Western']

user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

col_names = other_cols + user_cols +  genre_cols


# Dataset de filmes
movies = pd.read_csv('u.item', sep='|', encoding='latin-1', header=None, names=other_cols + genre_cols)
movies = movies.rename(columns={'movie_id':'item_id'})
movies = movies.drop(columns=['video_release_date', 'IMDb_URL'])

# Dataset de notas
ratings = pd.read_csv('u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

# Dataset de usuarios
users = pd.read_csv('u.user', sep='|', encoding='latin-1', header=None,
                    names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])

# Merge
df = ratings.merge(movies, on='item_id', how='left').merge(users, on='user_id', how='left')

df

,user_id,item_id,rating,timestamp,movie_title,release_date,Unknown,Action,Adventure,Animation,...,Mystery,Romance,Sci-Fi,Thriller,War,Western,age,gender,occupation,zip_code
0,196,242,3,881250949,Kolya (1996),24-Jan-1997,0,0,0,0,...,0,0,0,0,0,0,49,M,writer,55105
1,186,302,3,891717742,L.A. Confidential (1997),01-Jan-1997,0,0,0,0,...,1,0,0,1,0,0,39,F,executive,00000
2,22,377,1,878887116,Heavyweights (1994),01-Jan-1994,0,0,0,0,...,0,0,0,0,0,0,25,M,writer,40206
3,244,51,2,880606923,Legends of the Fall (1994),01-Jan-1994,0,0,0,0,...,0,1,0,0,1,1,28,M,technician,80525
4,166,346,1,886397596,Jackie Brown (1997),01-Jan-1997,0,0,0,0,...,0,0,0,0,0,0,47,M,educator,55113
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,880,476,3,880175444,"First Wives Club, The (1996)",14-Sep-1996,0,0,0,0,...,0,0,0,0,0,0,13,M,student,83702
99996,716,204,5,879795543,Back to the Future (1985),01-Jan-1985,0,0,0,0,...,0,0,1,0,0,0,36,F,administrator,44265
99997,276,1090,1,874795795,Sliver (1993),01-Jan-1993,0,0,0,0,...,0,0,0,1,0,0,21,M,student,95064
99998,13,225,2,882399156,101 Dalmatians (1996),27-Nov-1996,0,0,0,0,...,0,0,0,0,0,0,47,M,educator,29206


In [4]:
mask = df.isna().any(axis=1)
df[mask]

,user_id,item_id,rating,timestamp,movie_title,release_date,Unknown,Action,Adventure,Animation,...,Mystery,Romance,Sci-Fi,Thriller,War,Western,age,gender,occupation,zip_code
2172,130,267,5,875801239,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,20,M,none,60115
3781,5,267,4,875635064,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,33,F,other,15213
7245,268,267,3,875742077,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,24,M,engineer,19422
12475,297,267,3,875409139,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,29,F,educator,98103
14756,319,267,4,875707690,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,38,M,programmer,22030
15292,1,267,4,875692955,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,24,M,technician,85711
49295,532,267,3,875441348,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,20,M,student,92705
93523,833,267,1,875655669,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,34,M,writer,90019
99723,422,267,4,875655986,unknown,NaN,1,0,0,0,...,0,0,0,0,0,0,26,M,entertainment,94533


In [5]:
# Como tem apenas 9 instancias com valores NaN, podemos retira-las sem comprometer os dados
df = df.dropna()

In [6]:
def extract_year(release_date):
    try:
        return int(release_date.strip()[-4:])
    except:
        return np.nan

# Extraindo os anos do 'release_date' de cada filme e criando um 'year_bucket'
# com a decada em que o filme foi lancado para usar como feature
movies['year'] = movies['release_date'].apply(extract_year)
movies['year_bucket'] = pd.cut(movies['year'],
                               bins=[0,1970,1980,1990,2000,2010,2020,3000],
                               labels=['<1970','70s','80s','90s','2000s','2010s','2020+'])

# Criando um 'age_bucket' como feature do usuario
users['age_bucket'] = pd.cut(users['age'],
                             bins=[0,18,25,35,45,55,100],
                             labels=['<18','18-24','25-34','35-44','45-54','55+'])

# Tirando features dos filmes que nao sao importantes
movies_small = movies.drop(columns=['movie_title','release_date'])
df = ratings.merge(movies_small, on='item_id', how='left').merge(users, on='user_id', how='left')
df = df.dropna()


In [7]:
# Transformando a feature 'zip_code' em estados para usar como feature
nomi = pgeocode.Nominatim('us')
tqdm.pandas()

df['state'] = df['zip_code'].progress_apply(lambda x: nomi.query_postal_code(x).state_code)
df = df.drop(columns=['zip_code'])

100%|██████████| 99991/99991 [17:22<00:00, 95.92it/s] 


In [8]:
# Fazendo o split de treino (80%) e test (20%) nos dados
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test  = df.iloc[idx_test].reset_index(drop=True)

In [9]:
def attach_user_item_stats(df_train: pd.DataFrame,
                           df_target: pd.DataFrame) -> pd.DataFrame:
    '''
    Calcula a media de ratings dos usuarios e dos itens no conjunto
    de treino para usar como feature. Como nao esta sendo usado os
    dados do conjunto de teste, nao ha data leak.
    '''

    # Media global das avaliacoes no conjunto de treino
    gavg = df_train['rating'].mean()

    # Media de ratings por usuario
    u_stats = (
        df_train.groupby('user_id')['rating']
                 .agg(user_avg='mean', user_cnt='size')
                 .reset_index()
    )

    # Media de ratings por item
    i_stats = (
        df_train.groupby('item_id')['rating']
                 .agg(item_avg='mean', item_cnt='size')
                 .reset_index()
    )

    # DataFrame de saida
    out = (df_target
           .merge(u_stats, on='user_id', how='left')
           .merge(i_stats, on='item_id', how='left'))

    # Coloca os as medias dos usuarios e dos itens no df de saida
    # preenchendo NaNs com a media global
    out['user_avg'] = out['user_avg'].fillna(gavg)
    out['item_avg'] = out['item_avg'].fillna(gavg)

    return out

df_train_ex = attach_user_item_stats(df_train, df_train)
df_test_ex  = attach_user_item_stats(df_train, df_test)

In [10]:
feature_names = genre_cols + ['gender', 'occupation', 'state', 'year_bucket', 'age_bucket']

In [11]:
def row_to_feat(row : pd.Series) -> dict:
    '''
    Pega uma linha do DataFrame e transforma em um dicionario de features
    (necessario para o FM).
    '''
    feat = {}

    # Trat ID como one-hot (necessario para FM)
    feat[f'user_{int(row.user_id)}'] = 1
    feat[f'item_{int(row.item_id)}'] = 1

    # Generos dos filmes
    for g in genre_cols:
        if row.get(g) == 1:
            feat[f'genre_{g}'] = 1

    # Features categoricas do user
    if pd.notnull(row.get('gender')):
        feat[f'gender_{row.gender}'] = 1
    if pd.notnull(row.get('occupation')):
        feat[f'occupation_{row.occupation}'] = 1
    if pd.notnull(row.get('state')):
        feat[f'{str(row.state)}'] = 1
    if pd.notnull(row.get('year_bucket')):
        feat[f'year_{row.year_bucket}'] = 1
    if pd.notnull(row.get('age_bucket')):
        feat[f'age_{row.age_bucket}'] = 1

    # Features continuas
    feat['user_avg'] = float(row.user_avg)
    feat['item_avg'] = float(row.item_avg)

    return feat

In [12]:
# Cria os dicionarios dos splits de treino e test
dicts_train = df_train_ex.apply(row_to_feat, axis=1).tolist()
dicts_test  = df_test_ex.apply(row_to_feat, axis=1).tolist()

# Vetorizacao one-hot pra matriz esparca (necessario para FM)
vec = DictVectorizer(sparse=True)
X_train = vec.fit_transform(dicts_train).astype(np.float32)
X_test  = vec.transform(dicts_test).astype(np.float32)

# Pega o rotulo
y_train = df_train_ex['rating'].values.astype(np.float32)
y_test  = df_test_ex['rating'].values.astype(np.float32)

### FM

In [13]:
# Instancia o modelo
fm = FMRegression(n_iter=200, init_stdev=0.1, rank=4,
                  l2_reg_w=0.1, l2_reg_V=1.5)
fm.fit(X_train, y_train)

# Previsoes
y_pred = fm.predict(X_test)

# Calcula o RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE (test): {rmse:.4f}')

RMSE (test): 0.9427


### EBM

In [14]:
# Separando as variaveis que serao usadas pelo EBM
ebm_cats = ['year_bucket', 'age_bucket', 'gender', 'occupation', 'state']
ebm_cont = ['user_avg', 'item_avg']
ebm_cols = genre_cols + ebm_cats + ebm_cont


In [15]:
def make_ebm_frame(df_in, cols, cont_fill=None):
    """
    Funcao que ajeita as variaveis para o EBM
    """
    df_out = df_in.copy()
    for c in cols:
        if c in ebm_cats:
            df_out[c] = df_out[c].astype(str).fillna('UNK').replace({'nan':'UNK'})
        elif c in ebm_cont:
            df_out[c] = pd.to_numeric(df_out[c], errors='coerce')
        else:
            df_out[c] = pd.to_numeric(df_out[c], errors='coerce').fillna(0.0)

    # Preenchimento estável das contínuas (use médias do TREINO)
    if cont_fill is not None:
        for c in ebm_cont:
            if c in df_out.columns:
                df_out[c] = df_out[c].fillna(cont_fill.get(c, float(df_out[c].mean())))
    else:
        for c in ebm_cont:
            if c in df_out.columns:
                df_out[c] = df_out[c].fillna(float(df_out[c].mean()))

    return df_out[cols]


In [16]:
# medias das contínuas no TREINO para preencher o TESTE
_cont_fill = df_train_ex[ebm_cont].mean(numeric_only=True).to_dict()

# Deixando os dados dos splits de treino e teste no formato adequado
X_ebm_train = make_ebm_frame(df_train_ex, ebm_cols, cont_fill=_cont_fill)
X_ebm_test  = make_ebm_frame(df_test_ex,  ebm_cols, cont_fill=_cont_fill)


In [17]:
# Criando as variaveis alvo com o predict do FM
y_fm_train = fm.predict(X_train)
y_fm_test  = fm.predict(X_test)

In [18]:
# Instanciando e treinando o EBM
ebm = ExplainableBoostingRegressor(
    interactions=10,
    max_rounds=5000,
    random_state=42
)
ebm.fit(X_ebm_train, y_fm_train)

ExplainableBoostingRegressor(interactions=10, max_rounds=5000)

In [19]:
# Pegandoos predicts do EBM
pred_test  = ebm.predict(X_ebm_test)

# Calculando metricas
r2_test    = r2_score(y_fm_test,  pred_test)

print(f"R^2  test: {r2_test:.4f}")

# Fazendo display das explicacoes globais do modelo
global_exp = ebm.explain_global(name="EBM surrogate of FM")
imp = pd.DataFrame({
    "term":  global_exp.data()['names'],
    "score": global_exp.data()['scores']
}).sort_values("score", ascending=False)
display(imp)

R^2  test: 0.6344


,term,score
25,item_avg,0.380684
24,user_avg,0.283182
23,state,0.043861
35,user_avg & item_avg,0.038148
20,age_bucket,0.030918
22,occupation,0.030496
34,occupation & item_avg,0.022935
33,gender & item_avg,0.018282
30,Sci-Fi & gender,0.011831
21,gender,0.011260


In [20]:
def evaluate_ebm_local_stability(
    ebm,
    X_ebm_train: pd.DataFrame,
    X_ebm_test: pd.DataFrame,
    ebm_cont: list,
    row_index: int = 0,
    n: int = 10,
    noise_frac: float = 0.02,
    clip_to_train: bool = True,
    random_state: int = 42
):

    # Linha base (1xD) e copias para perturbar
    row0 = X_ebm_test.iloc[row_index:row_index+1].copy()
    Xp = pd.concat([row0]*n, ignore_index=True)

    # Escala de ruido pelas stds do TREINO
    stds = X_ebm_train[ebm_cont].std(ddof=0).replace(0, 1e-12)
    mins = X_ebm_train[ebm_cont].min() if clip_to_train else None
    maxs = X_ebm_train[ebm_cont].max() if clip_to_train else None

    # Aplica jitter apenas nas contínuas
    for c in ebm_cont:
        noise = rng.normal(loc=0.0, scale=float(stds[c]) * noise_frac, size=n)
        Xp[c] = pd.to_numeric(Xp[c], errors='coerce').astype(float) + noise
        if clip_to_train:
            Xp[c] = Xp[c].clip(lower=mins[c], upper=maxs[c])

    # Previsões
    base_pred = float(ebm.predict(row0)[0])
    preds = ebm.predict(Xp).astype(float)

    # Desvio padrao
    pred_std   = float(preds.std(ddof=0))

    # Sensibilidade local: quanto a saida varia por unidade de perturbacao
    deltas = (Xp[ebm_cont].values - row0[ebm_cont].values)
    l2 = np.linalg.norm(deltas, axis=1)
    avg_l2 = float(l2.mean())
    local_sensitivity = float(pred_std / (avg_l2 + 1e-12))

    report = {
        "row_index": int(row_index),
        "n_perturb": int(n),
        "noise_frac": float(noise_frac),
        "base_pred": base_pred,
        "pred_std": pred_std,
        "local_sensitivity": local_sensitivity
    }

    return report, Xp.assign(pred=preds)


In [21]:
report, samples = evaluate_ebm_local_stability(
    ebm, X_ebm_train, X_ebm_test, ebm_cont,
    row_index=0, n=100, noise_frac=0.02
)
print(report)
# display(samples[ebm_cont + ['pred']])

{'row_index': 0, 'n_perturb': 100, 'noise_frac': 0.02, 'base_pred': 3.4213555326403338, 'pred_std': 0.07362362129134578, 'local_sensitivity': 6.519525799942159}
